# Software Developer Salary Prediction Using Linear Regression

**Objective:** Predict software developer salaries using Linear Regression and create user interfaces with Gradio and Flask.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
data = pd.read_csv("train.csv")

In [3]:
print("Dataset shape:", data.shape)
print("\nColumns:", data.columns.tolist())
print("\nMissing values:\n", data.isnull().sum())

data.head()

Dataset shape: (40000, 7)

Columns: ['experience', 'country', 'education', 'languages', 'frameworks', 'company_size', 'salary_usd']

Missing values:
 experience      0
country         0
education       0
languages       0
frameworks      0
company_size    0
salary_usd      0
dtype: int64


,experience,country,education,languages,frameworks,company_size,salary_usd
0,34,Germany,Masters,"Rust, JavaScript","Vue, ASP.NET",201-1000,172292
1,9,UK,Masters,"C#, JavaScript","Ruby on Rails, Spring",201-1000,82311
2,8,USA,Some College,"C++, Go","Vue, React",5000+,109637
3,29,USA,Masters,"C++, Ruby","Spring, Angular",11-50,214386
4,7,USA,Bachelors,"C#, Rust","React, Express",1-10,107584


### Data Preparation
We select the input features and salary target, then convert categorical data into numeric columns.

In [4]:
X = data.drop("salary_usd", axis=1)
y = data["salary_usd"]

In [5]:
X = pd.get_dummies(X, dtype=int)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: (40000, 222)
Target shape: (40000,)


### Train-Test Split
We use 80% of the data for training and 20% for testing.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (32000, 222)
Testing data: (8000, 222)


### Model Training and Evaluation
We train the Linear Regression model, predict salaries, and evaluate its performance using MAE, MSE, RMSE, and R² score.

In [8]:
model = LinearRegression()

In [9]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [10]:
print("MAE:", round(mean_absolute_error(y_test, y_pred), 2))
print("MSE:", round(mean_squared_error(y_test, y_pred), 2))
print("RMSE:", round(mean_squared_error(y_test, y_pred) ** 0.5, 2))
print("R2 Score:", round(r2_score(y_test, y_pred), 4))

MAE: 13431.0
MSE: 263693324.83
RMSE: 16238.64
R2 Score: 0.8775


## Gradio Interface
We create a web interface where users enter developer details and get a predicted annual salary.

In [11]:
def predict_salary(experience, country, education, languages, frameworks, company_size):

    input_data = pd.DataFrame([{
        "experience": experience,
        "country": country,
        "education": education,
        "languages": languages,
        "frameworks": frameworks,
        "company_size": company_size
    }])

    input_data = pd.get_dummies(input_data, dtype=int)
    input_data = input_data.reindex(columns=X.columns, fill_value=0)

    prediction = model.predict(input_data)[0]

    return round(float(prediction), 2)

In [12]:
import gradio as gr

demo = gr.Interface(
    fn=predict_salary,
    inputs=[
        gr.Number(label="Years of Experience", value=5),
        gr.Dropdown(sorted(data["country"].unique().tolist()), label="Country"),
        gr.Dropdown(sorted(data["education"].unique().tolist()), label="Education"),
        gr.Dropdown(sorted(data["languages"].unique().tolist()), label="Programming Languages"),
        gr.Dropdown(sorted(data["frameworks"].unique().tolist()), label="Frameworks"),
        gr.Dropdown(sorted(data["company_size"].unique().tolist()), label="Company Size")
    ],
    outputs=gr.Number(label="Predicted Annual Salary (USD)"),
    title="Software Developer Salary Prediction",
    description="Enter developer details to predict the annual salary in USD."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.



## Flask Interface
We create a webpage using Flask where users enter developer details and receive a predicted salary.

In [16]:
from flask import Flask, request, render_template_string
from threading import Thread

app = Flask(__name__)

html_page = """
<!DOCTYPE html>
<html>
<head>
    <title>Software Developer Salary Prediction</title>
</head>
<body>
    <h2>Software Developer Salary Prediction</h2>

    <form method="POST">
        <label>Years of Experience:</label>
        <input type="number" name="experience" min="0" max="40" value="5" required>
        <br><br>

        <label>Country:</label>
        <select name="country">
            {% for item in countries %}
            <option value="{{ item }}">{{ item }}</option>
            {% endfor %}
        </select>
        <br><br>

        <label>Education:</label>
        <select name="education">
            {% for item in educations %}
            <option value="{{ item }}">{{ item }}</option>
            {% endfor %}
        </select>
        <br><br>

        <label>Programming Languages:</label>
        <select name="languages">
            {% for item in languages %}
            <option value="{{ item }}">{{ item }}</option>
            {% endfor %}
        </select>
        <br><br>

        <label>Frameworks:</label>
        <select name="frameworks">
            {% for item in frameworks %}
            <option value="{{ item }}">{{ item }}</option>
            {% endfor %}
        </select>
        <br><br>

        <label>Company Size:</label>
        <select name="company_size">
            {% for item in company_sizes %}
            <option value="{{ item }}">{{ item }}</option>
            {% endfor %}
        </select>
        <br><br>

        <button type="submit">Predict Salary</button>
    </form>

    {% if prediction is not none %}
        <h3>Predicted Annual Salary (USD): {{ prediction }}</h3>
    {% endif %}
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():
    prediction = None

    if request.method == "POST":
        prediction = predict_salary(
            float(request.form["experience"]),
            request.form["country"],
            request.form["education"],
            request.form["languages"],
            request.form["frameworks"],
            request.form["company_size"]
        )

    return render_template_string(
        html_page,
        prediction=prediction,
        countries=sorted(data["country"].unique().tolist()),
        educations=sorted(data["education"].unique().tolist()),
        languages=sorted(data["languages"].unique().tolist()),
        frameworks=sorted(data["frameworks"].unique().tolist()),
        company_sizes=sorted(data["company_size"].unique().tolist())
    )

In [17]:
print(data.columns.tolist())
print(data.head(2))
print("Target:", y.name)

['experience', 'country', 'education', 'languages', 'frameworks', 'company_size', 'salary_usd']
   experience  country education         languages             frameworks  \
0          34  Germany   Masters  Rust, JavaScript           Vue, ASP.NET   
1           9       UK   Masters    C#, JavaScript  Ruby on Rails, Spring   

  company_size  salary_usd  
0     201-1000      172292  
1     201-1000       82311  
Target: salary_usd


In [18]:
from threading import Thread

flask_thread = Thread(
    target=lambda: app.run(
        host="127.0.0.1",
        port=5001,
        debug=False,
        use_reloader=False
    ),
    daemon=True
)

flask_thread.start()

print("Salary Prediction Flask server started.")
print("Open http://127.0.0.1:5001")

Salary Prediction Flask server started.
Open http://127.0.0.1:5001
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [12/Oct/2026 11:56:34] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Oct/2026 11:56:35] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [12/Oct/2026 11:56:51] "POST / HTTP/1.1" 200 -
